# ДЗ 2.3. SCD2 и JOIN по расписанию

- набор данных: `demo` (6 000 событий, 100 пользователей, seed=42, 1 января - 1 февраля 2026). 
- контрольные примеры JOIN: на `fixtures`.

Расчёты выполняет только DAG, ноутбук только проверяет его результаты.

**Пути в S3**
- вход: `s3://hw23/raw/demo/events/*.csv`, `s3://hw23/raw/demo/campaigns.csv`;
- `s3://hw23/results/demo/user_scd2/snapshot_date=D/`;
- `s3://hw23/results/demo/campaign_recipients/calculation_date=D/`;
- копия до повторного запуска: `s3://hw23/results/demo/_before_rerun/`.

Партиция (Hive-стиль): `snapshot_date` / `calculation_date` хранятся в имени каталога и при чтении таблицы появляются как колонки типа `date`.

## 0. Подготовка

In [1]:
import sys, time, subprocess
from datetime import datetime, timedelta, timezone
sys.path.append("/opt/airflow/jobs")
from pyspark.sql import functions as F
from airflow.models import DagModel, DagRun
from airflow.utils.session import create_session

from common import RESULTS, META, create_spark, read_events, read_campaigns, utc_ts, with_meta, write_partition
from task1_user_scd2 import build_scd2
from task2_campaign_recipients import recipients

airflow = lambda cmd: print(subprocess.run(f"airflow {cmd}", shell=True, capture_output=True, text=True).stdout)   # CLI Airflow
DAG_ID, D1 = "hw23_scd2_campaigns", "2026-01-23"
SCD2, REC, BAK = f"{RESULTS}/user_scd2", f"{RESULTS}/campaign_recipients", f"{RESULTS}/_before_rerun"

## 1. DAG и три запуска по расписанию

In [2]:
!cat /opt/airflow/dags/hw23_scd2_campaigns.py

from datetime import datetime, timedelta, timezone

from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.timetables.interval import CronDataIntervalTimetable


def build_user_scd2(**ctx):
    from task1_user_scd2 import main          # /opt/airflow/jobs в PYTHONPATH; pyspark импортируется только в задаче
    return main(**ctx)                  # в XCom уходит только путь к снимку


def build_campaign_recipients(scd2_path, **ctx):
    from task2_campaign_recipients import main
    return main(scd2_path, **ctx)

# ДЗ 2.3: build_user_scd2 -> build_campaign_recipients, ежедневно в 00:00 UTC за сутки [start, end)
with DAG(
    dag_id="hw23_scd2_campaigns",
    # ежедневно в 00:00 UTC, каждый запуск обрабатывает завершённые сутки [data_interval_start, data_interval_end).
    # CronDataIntervalTimetable даёт интервал ровно в сутки
    schedule=CronDataIntervalTimetable("0 0 * * *", timezone="UTC"),
    # c 23 по 25 января 2026
    start_date=datetime(2026, 

Включаем DAG и ждём, пока scheduler выполнит три запуска (в Airflow DAG по умолчанию выключен).

In [3]:
DagModel.get_dagmodel(DAG_ID).set_is_paused(is_paused=False)   # = airflow dags unpause

In [4]:
def airflow_runs():
    # Экспорт из БД Airflow: тип запуска, интервал, состояние и номер попытки каждой задачи
    with create_session() as s:
        rows = []
        for dr in s.query(DagRun).filter(DagRun.dag_id == DAG_ID).order_by(DagRun.data_interval_start):
            ti = {t.task_id: f"{t.state} (try {t.try_number})" for t in dr.get_task_instances(session=s)}
            ti = {t: ti.get(t) for t in ("build_user_scd2", "build_campaign_recipients")}
            rows.append({"run_id": dr.run_id, 
                         "run_type": dr.run_type, 
                         "state": dr.state,
                         "interval_start": str(dr.data_interval_start), 
                         "interval_end": str(dr.data_interval_end),
                         "hours": (dr.data_interval_end - dr.data_interval_start) / timedelta(hours=1), **ti})
    return rows

In [5]:
def print_table(rows):
    w = {c: max(len(c), *(len(str(r[c])) for r in rows)) for c in rows[0]}
    for r in [dict(zip(w, w))] + rows:
        print(" | ".join(str(r[c]).ljust(w[c]) for c in w))

def wait_runs(n=3):
    while True:
        rows = airflow_runs()
        if any(r["state"] == "failed" for r in rows):
            raise RuntimeError(rows)
        if len(rows) == n and all(r["state"] == "success" for r in rows):
            return print_table(rows)
        time.sleep(15)

wait_runs()

run_id                               | run_type  | state   | interval_start            | interval_end              | hours | build_user_scd2 | build_campaign_recipients
scheduled__2026-01-23T00:00:00+00:00 | scheduled | success | 2026-01-23 00:00:00+00:00 | 2026-01-24 00:00:00+00:00 | 24.0  | success (try 1) | success (try 1)          
scheduled__2026-01-24T00:00:00+00:00 | scheduled | success | 2026-01-24 00:00:00+00:00 | 2026-01-25 00:00:00+00:00 | 24.0  | success (try 1) | success (try 1)          
scheduled__2026-01-25T00:00:00+00:00 | scheduled | success | 2026-01-25 00:00:00+00:00 | 2026-01-26 00:00:00+00:00 | 24.0  | success (try 1) | success (try 1)          


Три запуска типа `scheduled` с интервалами за 23, 24 и 25 января, каждый ровно 24 часа, обе задачи `success`.

## 2. Исходные данные
Диапазон `event_time` покрывает январские рассылки; за три даты расписания: 5 рассылок.

In [6]:
spark = create_spark("hw23_report")   # Spark только для чтения результатов и проверок
sql = lambda q, n=50: spark.sql(q).show(n, truncate=False)

events, campaigns = read_events(spark), read_campaigns(spark)

events.agg(F.count("*").alias("events"), 
           F.countDistinct("user_id").alias("users"),
           F.min("event_time").alias("min_event_time"), 
           F.max("event_time").alias("max_event_time")).show()

campaigns.filter("send_at >= '2026-01-23' and send_at < '2026-01-26'").orderBy("send_at").show()

26/11/19 15:05:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 15:05:39 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+------+-----+-------------------+-------------------+
|events|users|     min_event_time|     max_event_time|
+------+-----+-------------------+-------------------+
|  6000|  100|2026-01-01 00:10:55|2026-02-01 08:29:58|
+------+-----+-------------------+-------------------+

+-----------+---------------+-------------------+-------------+
|campaign_id|   email_domain|            send_at|campaign_name|
+-----------+---------------+-------------------+-------------+
|         16|   post.example|2026-01-23 03:02:59|Campaign_0016|
|         11|   mail.example|2026-01-23 07:18:34|Campaign_0011|
|         23|    example.com|2026-01-23 11:37:34|Campaign_0023|
|         21|  inbox.example|2026-01-24 05:07:48|Campaign_0021|
|         19|webmail.example|2026-01-25 05:32:10|Campaign_0019|
+-----------+---------------+-------------------+-------------+



## 3. Снимки SCD2

Проверки по каждому `snapshot_date`:
- `users_in_events`: пользователи с событиями до конца интервала, должно совпасть с `users`;
- `dup_keys`: дубли ключа `(user_id, valid_from)`;
- `bad_intervals`: `valid_to ≤ valid_from` или `valid_to` предыдущей версии ≠ `valid_from` следующей (пересечения/разрывы);
- `bad_current`: пользователи, у которых не ровно одна версия с `valid_to IS NULL AND is_current`, плюс строки, где `is_current` не соответствует `valid_to`;
- `same_as_prev`: версия с теми же ФИО, почтой и логином, что предыдущая (лишняя версия);
- `future`: `valid_from ≥ data_interval_end`; `nulls`: NULL в бизнес-полях, кроме `valid_to`.

In [7]:
spark.read.parquet(SCD2).createOrReplaceTempView("user_scd2")
events.createOrReplaceTempView("events")
spark.table("user_scd2").printSchema()
sql('''
with v as (
    select 
        *, 
        lag(valid_to) over w as prev_to,
        -- проверка «нет ли лишней версии»
        lag(user_fio) over w <=> user_fio and lag(user_email) over w <=> user_email and lag(user_login) over w <=> user_login as same_as_prev
    from user_scd2 window w as (partition by snapshot_date, user_id order by valid_from)),
current_versions as (
    -- сколько текущих версий у каждого пользователя в снимке (должна быть ровно одна)
    select 
        snapshot_date, 
        user_id, 
        sum(cast(is_current and valid_to is null as int)) as n
    from user_scd2 
    group by 1, 2),
cur as (
    select 
        snapshot_date, 
        count_if(n <> 1) as bad_users 
    from current_versions 
    group by 1),
snapshots as (
    select distinct 
        snapshot_date 
    from user_scd2),
ev as (
    -- пользователи, у которых есть события до конца интервала снимка
    select 
        s.snapshot_date, 
        count(distinct e.user_id) as users_in_events
    from snapshots s 
    join events e on e.event_time < date_add(s.snapshot_date, 1)
    group by 1),
agg as (
    select 
        snapshot_date, 
        count(*) as rows, 
        count(distinct user_id) as users,
        count(*) - count(distinct user_id, valid_from) as dup_keys,
        count_if(valid_to <= valid_from or (prev_to is not null and prev_to <> valid_from)) as bad_intervals,
        count_if(is_current <> (valid_to is null)) as bad_flag,
        count_if(same_as_prev) as same_as_prev,
        count_if(valid_from >= date_add(snapshot_date, 1)) as future,
        count_if(user_id is null or user_fio is null or user_email is null or user_login is null
              or valid_from is null or is_current is null) as nulls,
        max(valid_from) as max_valid_from
    from v 
    group by 1)
    
select 
    snapshot_date, 
    rows, 
    users, 
    users_in_events, 
    dup_keys, 
    bad_intervals, 
    bad_users + bad_flag as bad_current,
    same_as_prev, 
    future, nulls, 
    max_valid_from
from agg 
join ev using (snapshot_date) 
join cur using (snapshot_date) 
order by snapshot_date''')

root
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- _dag_id: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _task_id: string (nullable = true)
 |-- _data_interval_start: timestamp (nullable = true)
 |-- _data_interval_end: timestamp (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)
 |-- snapshot_date: date (nullable = true)



+-------------+----+-----+---------------+--------+-------------+-----------+------------+------+-----+-------------------+
|snapshot_date|rows|users|users_in_events|dup_keys|bad_intervals|bad_current|same_as_prev|future|nulls|max_valid_from     |
+-------------+----+-----+---------------+--------+-------------+-----------+------------+------+-----+-------------------+
|2026-01-23   |961 |100  |100            |0       |0            |0          |0           |0     |0    |2026-01-23 23:17:35|
|2026-01-24   |995 |100  |100            |0       |0            |0          |0           |0     |0    |2026-01-24 23:41:16|
|2026-01-25   |1036|100  |100            |0       |0            |0          |0           |0     |0    |2026-01-25 23:25:27|
+-------------+----+-----+---------------+--------+-------------+-----------+------------+------+-----+-------------------+



Все проверки: 0, число пользователей в каждом снимке совпадает с числом пользователей в событиях до конца интервала, `max_valid_from` меньше конца суток. С каждым днём снимок растёт за счёт новых версий.

## 4. Витрина campaign_recipients

Количество получателей по рассылкам и проверки по каждой дате:
- `dup_pairs`: дубли `(campaign_id, user_id)`;
- `bad_domain`: домен почты версии ≠ `email_domain` (без учёта регистра);
- `bad_interval`: `send_at` вне `[user_valid_from, user_valid_to)`;
- `wrong_day`: `send_at` вне интервала запуска;
- `diff_vs_events`: сравнение с независимым расчётом прямо по событиям (состояние профиля на `send_at` = последнее событие с `event_time ≤ send_at`), `exceptAll` в обе стороны по `(campaign_id, user_id)`.

In [8]:
spark.read.parquet(REC).createOrReplaceTempView("campaign_recipients")
campaigns.createOrReplaceTempView("campaigns")
spark.table("campaign_recipients").printSchema()
sql('''
select 
    calculation_date, 
    campaign_id, 
    campaign_name, 
    send_at, 
    email_domain, 
    count(*) as recipients
from campaign_recipients 
group by all 
order by calculation_date, campaign_id''')

root
 |-- campaign_id: long (nullable = true)
 |-- campaign_name: string (nullable = true)
 |-- send_at: timestamp (nullable = true)
 |-- email_domain: string (nullable = true)
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- user_valid_from: timestamp (nullable = true)
 |-- user_valid_to: timestamp (nullable = true)
 |-- _dag_id: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _task_id: string (nullable = true)
 |-- _data_interval_start: timestamp (nullable = true)
 |-- _data_interval_end: timestamp (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)
 |-- calculation_date: date (nullable = true)

+----------------+-----------+-------------+-------------------+---------------+----------+
|calculation_date|campaign_id|campaign_name|send_at            |email_domain   |recipients|
+----------------+-----------+-------------+------------------

In [9]:
# Независимый расчёт: последнее событие пользователя к моменту send_at, без SCD2
expected = spark.sql('''
with day_campaigns as (
    -- рассылки за три даты расписания
    select 
        campaign_id, 
        email_domain, 
        send_at 
    from campaigns 
    where send_at >= '2026-01-23' and send_at < '2026-01-26'),
events_before_send as (
    -- события пользователя до момента рассылки; rn = 1 -> последнее, т.е. состояние профиля на send_at
    select 
        c.campaign_id, 
        c.email_domain, 
        e.user_id, 
        e.user_email,
        row_number() over (partition by c.campaign_id, e.user_id order by e.event_time desc) as rn
    from day_campaigns c 
    join events e on e.event_time <= c.send_at)

select 
    campaign_id, 
    user_id 
from events_before_send 
where rn = 1 
  and lower(substring_index(user_email, '@', -1)) = lower(email_domain)''')

actual = spark.table("campaign_recipients").select("campaign_id", "user_id")
print("diff_vs_events:", actual.exceptAll(expected).count() + expected.exceptAll(actual).count(),
      "| строк в витрине:", actual.count(), "| ожидается:", expected.count())

sql('''
select 
    calculation_date, 
    count(*) as rows, 
    count(distinct user_id) as users,
    count(*) - count(distinct campaign_id, user_id) as dup_pairs,
    count_if(lower(substring_index(user_email, '@', -1)) <> lower(email_domain)) as bad_domain,
    count_if(not (user_valid_from <= send_at and (user_valid_to is null or send_at < user_valid_to))) as bad_interval,
    count_if(send_at < _data_interval_start or send_at >= _data_interval_end
        or to_date(send_at) <> calculation_date) as wrong_day
from campaign_recipients 
group by 1 order by 1''')

diff_vs_events: 0 | строк в витрине: 100 | ожидается: 100
+----------------+----+-----+---------+----------+------------+---------+
|calculation_date|rows|users|dup_pairs|bad_domain|bad_interval|wrong_day|
+----------------+----+-----+---------+----------+------------+---------+
|2026-01-23      |53  |53   |0        |0         |0           |0        |
|2026-01-24      |25  |25   |0        |0         |0           |0        |
|2026-01-25      |22  |22   |0        |0         |0           |0        |
+----------------+----+-----+---------+----------+------------+---------+



Дублей нет, домен и исторический интервал соответствуют каждой строке, а результат полностью совпадает с независимым расчётом по событиям.

### План JOIN и стратегия

Тот же запрос, что во второй задаче (план также печатается в логе задачи `build_campaign_recipients`):

In [10]:
day = campaigns.filter((F.col("send_at") >= f"{D1}") & (F.col("send_at") < "2026-01-24"))
recipients(spark.read.parquet(f"{SCD2}/snapshot_date={D1}"), day).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [campaign_id#10L, campaign_name#13, send_at#12, email_domain#11, user_id#963L, user_fio#964, user_email#965, user_login#966, valid_from#967 AS user_valid_from#1041, valid_to#968 AS user_valid_to#1042]
   +- BroadcastHashJoin [domain#989], [lower(email_domain#11)], Inner, BuildRight, ((valid_from#967 <= send_at#12) AND (isnull(valid_to#968) OR (send_at#12 < valid_to#968))), false
      :- Project [user_id#963L, user_fio#964, user_email#965, user_login#966, valid_from#967, valid_to#968, lower(substring_index(user_email#965, @, -1)) AS domain#989]
      :  +- Filter (isnotnull(lower(substring_index(user_email#965, @, -1))) AND isnotnull(valid_from#967))
      :     +- FileScan parquet [user_id#963L,user_fio#964,user_email#965,user_login#966,valid_from#967,valid_to#968] Batched: true, DataFilters: [isnotnull(lower(substring_index(user_email#965, @, -1))), isnotnull(valid_from#967)], Format: Parquet, Location: InMemoryFileIn

Spark выбрал **BroadcastHashJoin**: 

- ключ равенства: `lower(домен почты) = lower(email_domain)`,
- условие по интервалу `valid_from ≤ send_at < valid_to` проверяется как дополнительный фильтр той же операции.

Рассылок за сутки 1-3 строки, поэтому они рассылаются всем задачам целиком (`BroadcastExchange`), и снимок SCD2 не перемешивается (нет `Exchange hashpartitioning` на стороне SCD2).

Это вывод из ДЗ 2.2: со справочником из 25 рассылок Spark сам выбирал BroadcastHashJoin, и варианты без shuffle (auto/broadcast) были быстрее sort-merge и shuffle-hash. 

Здесь справочник ещё меньше, поэтому broadcast задан явно (`F.broadcast`), чтобы план не зависел от порога `autoBroadcastJoinThreshold`.

## 5. Контрольные примеры на fixtures

Тот же код (`build_scd2` и `recipients` из `jobs/`), что в задачах DAG, по суткам 22-27 января. Ожидаемые получатели определены вручную по событиям:

In [11]:
fx_events, fx_campaigns = read_events(spark, "s3a://hw23/raw/fixtures"), read_campaigns(spark, "s3a://hw23/raw/fixtures")
day_bounds = lambda d: (datetime.fromisoformat(d).replace(tzinfo=timezone.utc),
                        datetime.fromisoformat(d).replace(tzinfo=timezone.utc) + timedelta(days=1))

def fx_day(d):
    # повторяет две задачи DAG для суток d
    start, end = day_bounds(d)
    snapshot = build_scd2(fx_events.filter(F.col("event_time") < utc_ts(end)))
    day = fx_campaigns.filter((F.col("send_at") >= utc_ts(start)) & (F.col("send_at") < utc_ts(end)))
    return snapshot, recipients(snapshot, day)

print("История Alice (1001) в снимке за 2026-01-25:")
fx_day("2026-01-25")[0].filter("user_id = 1001").orderBy("valid_from").show(truncate=False)

История Alice (1001) в снимке за 2026-01-25:
+-------+----------+------------------+----------+-------------------+-------------------+----------+
|user_id|user_fio  |user_email        |user_login|valid_from         |valid_to           |is_current|
+-------+----------+------------------+----------+-------------------+-------------------+----------+
|1001   |Alice One |alice@example.com |alice     |2026-01-22 08:00:00|2026-01-23 12:00:00|false     |
|1001   |Alice One |alice@mail.example|alice     |2026-01-23 12:00:00|2026-01-24 09:00:00|false     |
|1001   |Alicia One|alice@mail.example|alice     |2026-01-24 09:00:00|2026-01-25 12:00:00|false     |
|1001   |Alicia One|alice@mail.example|alice_new |2026-01-25 12:00:00|2026-01-25 18:00:00|false     |
|1001   |Alicia One|alice@mail.example|alice     |2026-01-25 18:00:00|NULL               |true      |
+-------+----------+------------------+----------+-------------------+-------------------+----------+



5 версий: 
- повторы 23.01 10:00 и 25.01 09:00 версий не создают,
- возврат логина `alice_new → alice` в 18:00 - новая версия,
- событие 26.01 (будущее) в снимок не попало, последняя версия открыта.

In [12]:
from functools import reduce
fx = reduce(lambda a, b: a.unionByName(b), [fx_day(f"2026-01-{d}")[1] for d in range(22, 28)])
fx.orderBy("campaign_id", "user_id").select("campaign_id", "campaign_name", "send_at", "user_id", "user_email",
                                            "user_valid_from", "user_valid_to").show(30, truncate=False)

+-----------+------------------------------+-------------------+-------+------------------+-------------------+-------------------+
|campaign_id|campaign_name                 |send_at            |user_id|user_email        |user_valid_from    |user_valid_to      |
+-----------+------------------------------+-------------------+-------+------------------+-------------------+-------------------+
|101        |Before_email_change           |2026-01-23 09:00:00|1001   |alice@example.com |2026-01-22 08:00:00|2026-01-23 12:00:00|
|102        |Old_domain_at_boundary        |2026-01-23 12:00:00|1002   |bob@example.com   |2026-01-23 12:00:00|NULL               |
|103        |New_domain_at_boundary        |2026-01-23 12:00:00|1001   |alice@mail.example|2026-01-23 12:00:00|NULL               |
|105        |Midnight_change               |2026-01-24 00:00:00|1001   |alice@mail.example|2026-01-23 12:00:00|2026-01-24 09:00:00|
|105        |Midnight_change               |2026-01-24 00:00:00|1002   |bob@

In [13]:
EXPECTED = {  # campaign_id: (что проверяем, ожидаемые user_id)
    101: ("старый домен до смены почты", {1001}),
    102: ("send_at = valid_from новой версии Bob; у Alice в 12:00 уже новый домен", {1002}),
    103: ("send_at = valid_from новой почты Alice", {1001}),
    104: ("за секунду до смены почты: новый домен ещё не действует", set()),
    105: ("смена почты Bob ровно в полночь", {1001, 1002}),
    106: ("первое событие Cara = send_at, домен Inbox.Example", {1003}),
    107: ("за секунду до первого события пользователя", set()),
    108: ("смена почты Bob в следующую полночь", {1002}),
    109: ("повтор события не создаёт версию (valid_from = 24.01 09:00)", {1001}),
    110: ("последняя секунда суток, первое событие Dan", {1004}),
    111: ("26.01 00:00 - Dan и первое событие Eve", {1004, 1005}),
    112: ("до появления Bob, у Alice example.com", {1001}),
    113: ("домена нет ни у кого", set()),
    114: ("пустой день 27.01", set()),
    115: ("смена только логина, домен рассылки в верхнем регистре", {1001}),
    116: ("возврат к прежнему состоянию A → B → A", {1001}),
}

In [14]:
got = {r.campaign_id: set(r.users) for r in fx.groupBy("campaign_id").agg(F.collect_set("user_id").alias("users")).collect()}
rows = [(cid, case, str(sorted(exp)), str(sorted(got.get(cid, set()))), exp == got.get(cid, set()))
        for cid, (case, exp) in EXPECTED.items()]

In [15]:
spark.createDataFrame(rows, "campaign_id INT, case STRING, expected STRING, actual STRING, ok BOOLEAN").show(truncate=False)
print("все совпали:", all(r[-1] for r in rows))

+-----------+----------------------------------------------------------------------+------------+------------+----+
|campaign_id|case                                                                  |expected    |actual      |ok  |
+-----------+----------------------------------------------------------------------+------------+------------+----+
|101        |старый домен до смены почты                                           |[1001]      |[1001]      |true|
|102        |send_at = valid_from новой версии Bob; у Alice в 12:00 уже новый домен|[1002]      |[1002]      |true|
|103        |send_at = valid_from новой почты Alice                                |[1001]      |[1001]      |true|
|104        |за секунду до смены почты: новый домен ещё не действует               |[]          |[]          |true|
|105        |смена почты Bob ровно в полночь                                       |[1001, 1002]|[1001, 1002]|true|
|106        |первое событие Cara = send_at, домен Inbox.Example         

**Пустой результат** (27.01: одна рассылка, получателей нет) пишется тем же `write_partition` -> получается читаемый Parquet с полной схемой и метаполями:

In [16]:
start, end = day_bounds("2026-01-27")
meta = {"dag_id": "fixtures_check", "run_id": "manual_check", "task_id": "build_campaign_recipients", "start": start, "end": end}
path = "s3a://hw23/results/fixtures/campaign_recipients/calculation_date=2026-01-27"
write_partition(with_meta(fx_day("2026-01-27")[1], meta), path)

In [17]:
empty = spark.read.parquet(path)
print("строк:", empty.count(), "| колонки:", empty.columns)

строк: 0 | колонки: ['campaign_id', 'campaign_name', 'send_at', 'email_domain', 'user_id', 'user_fio', 'user_email', 'user_login', 'user_valid_from', 'user_valid_to', '_dag_id', '_run_id', '_task_id', '_data_interval_start', '_data_interval_end', '_loaded_at']


## 6. Метаполя

Сводка по запускам для обеих таблиц, NULL в метаполях и согласованность задач одного запуска: одинаковые `_run_id` и интервал, дата партиции = дата начала интервала.

In [18]:
meta_summary = '''
select 
    '{t}' as tbl, 
    {d} as d, 
    _dag_id, 
    _run_id, 
    _task_id, 
    _data_interval_start, 
    _data_interval_end,
    count(*) as rows, 
    min(_loaded_at) as min_loaded_at, 
    max(_loaded_at) as max_loaded_at,
    count_if({nulls}) as null_meta
from {t} 
group by all'''
nulls = " or ".join(f"{c} is null" for c in META)

spark.sql(meta_summary.format(t="user_scd2", d="snapshot_date", nulls=nulls) + " union all " +
          meta_summary.format(t="campaign_recipients", d="calculation_date", nulls=nulls)
          ).createOrReplaceTempView("meta")

In [19]:
sql("select * from meta order by d, tbl desc")

+-------------------+----------+-------------------+------------------------------------+-------------------------+--------------------+-------------------+----+--------------------------+--------------------------+---------+
|tbl                |d         |_dag_id            |_run_id                             |_task_id                 |_data_interval_start|_data_interval_end |rows|min_loaded_at             |max_loaded_at             |null_meta|
+-------------------+----------+-------------------+------------------------------------+-------------------------+--------------------+-------------------+----+--------------------------+--------------------------+---------+
|user_scd2          |2026-01-23|hw23_scd2_campaigns|scheduled__2026-01-23T00:00:00+00:00|build_user_scd2          |2026-01-23 00:00:00 |2026-01-24 00:00:00|961 |2026-11-19 15:04:06.498696|2026-11-19 15:04:06.498696|0        |
|campaign_recipients|2026-01-23|hw23_scd2_campaigns|scheduled__2026-01-23T00:00:00+00:00|build_c

In [20]:
sql('''
with scd2_meta as (
    select 
        d, 
        _run_id, 
        _data_interval_start, 
        _data_interval_end, 
        max_loaded_at 
    from meta 
    where tbl = 'user_scd2'),
recipients_meta as (
    select 
        d, 
        _run_id, 
        _data_interval_start, 
        _data_interval_end, 
        min_loaded_at 
    from meta 
    where tbl = 'campaign_recipients')

select 
    s.d, 
    s._run_id = r._run_id as same_run_id,
    s._data_interval_start = r._data_interval_start and s._data_interval_end = r._data_interval_end as same_interval,
    to_date(s._data_interval_start) = s.d and r.d = s.d as date_matches,
    s.max_loaded_at < r.min_loaded_at as scd2_before_recipients
from scd2_meta s 
join recipients_meta r on s.d = r.d 
order by s.d''')

+----------+-----------+-------------+------------+----------------------+
|d         |same_run_id|same_interval|date_matches|scd2_before_recipients|
+----------+-----------+-------------+------------+----------------------+
|2026-01-23|true       |true         |true        |true                  |
|2026-01-24|true       |true         |true        |true                  |
|2026-01-25|true       |true         |true        |true                  |
+----------+-----------+-------------+------------+----------------------+



На каждую дату: ровно одна группа метаполей на таблицу (все строки снимка, включая старые версии, получили метаполя своего запуска), NULL нет, `_run_id` и интервал у двух задач одного запуска совпадают, снимок записан раньше витрины.

## 7. Повторный расчёт первого запуска

1. Сохраняем копии обеих таблиц за все три даты.
2. Clear обеих задач только запуска за 2026-01-23 -> scheduler выполняет их заново в том же DAG run.
3. Сравниваем до/после.

In [21]:
for t in ("user_scd2", "campaign_recipients"):
    # Обе таблицы целиком, за все три даты, копируются в s3://hw23/results/demo/_before_rerun/
    spark.read.parquet(f"{RESULTS}/{t}").write.mode("overwrite").parquet(f"{BAK}/{t}")

airflow(f"tasks clear {DAG_ID} -s 2026-01-23T00:00:00+00:00 -e 2026-01-23T00:00:00+00:00 -y")
time.sleep(5)
wait_runs()

[2026-11-19T15:06:08.946+0000] {dagbag.py:588} INFO - Filling up the DagBag from /opt/airflow/dags

run_id                               | run_type  | state   | interval_start            | interval_end              | hours | build_user_scd2 | build_campaign_recipients
scheduled__2026-01-23T00:00:00+00:00 | scheduled | success | 2026-01-23 00:00:00+00:00 | 2026-01-24 00:00:00+00:00 | 24.0  | success (try 2) | success (try 2)          
scheduled__2026-01-24T00:00:00+00:00 | scheduled | success | 2026-01-24 00:00:00+00:00 | 2026-01-25 00:00:00+00:00 | 24.0  | success (try 1) | success (try 1)          
scheduled__2026-01-25T00:00:00+00:00 | scheduled | success | 2026-01-25 00:00:00+00:00 | 2026-01-26 00:00:00+00:00 | 24.0  | success (try 1) | success (try 1)          


Обе задачи первого запуска выполнены второй попыткой (`try 2`) в том же DAG run, у остальных дат - по-прежнему первая.

In [22]:
for t, d in (("user_scd2", "snapshot_date"), ("campaign_recipients", "calculation_date")):
    before = spark.read.parquet(f"{BAK}/{t}")
    after = spark.read.parquet(f"{RESULTS}/{t}").select(before.columns)
    first = F.col(d) == D1
    b, a = before.filter(first).drop(*META), after.filter(first).drop(*META)     # D1: без шести метаполей
    bo, ao = before.filter(~first), after.filter(~first)                          # другие даты: все поля
    
    print(f"{t}: {D1} строк до/после {b.count()}/{a.count()}, до−после {b.exceptAll(a).count()}, после−до {a.exceptAll(b).count()} | "
          f"другие даты строк {bo.count()}/{ao.count()}, отличий (с метаполями) {bo.exceptAll(ao).count() + ao.exceptAll(bo).count()}")

    (before
     .filter(first)
     .select(F.lit("до").alias("when"), "_run_id", "_loaded_at")
     .distinct()
     .unionByName(after.filter(first).select(F.lit("после").alias("when"), "_run_id", "_loaded_at").distinct())
     .show(truncate=False))

user_scd2: 2026-01-23 строк до/после 961/961, до−после 0, после−до 0 | другие даты строк 2031/2031, отличий (с метаполями) 0
+-----+------------------------------------+--------------------------+
|when |_run_id                             |_loaded_at                |
+-----+------------------------------------+--------------------------+
|до   |scheduled__2026-01-23T00:00:00+00:00|2026-11-19 15:04:06.498696|
|после|scheduled__2026-01-23T00:00:00+00:00|2026-11-19 15:06:17.820501|
+-----+------------------------------------+--------------------------+

campaign_recipients: 2026-01-23 строк до/после 53/53, до−после 0, после−до 0 | другие даты строк 47/47, отличий (с метаполями) 0
+-----+------------------------------------+--------------------------+
|when |_run_id                             |_loaded_at                |
+-----+------------------------------------+--------------------------+
|до   |scheduled__2026-01-23T00:00:00+00:00|2026-11-19 15:04:22.715464|
|после|scheduled__2026-01

Бизнес-поля за 2026-01-23 до и после повтора совпадают полностью (`exceptAll` в обе стороны = 0), дублей не появилось, `_run_id` тот же (повтор внутри существующего DAG run), изменился только `_loaded_at`. Партиции 24 и 25 января не изменились даже в метаполях: перезаписывается только каталог своей даты.

## Итог
- DAG из двух зависимых задач, три запуска `scheduled` за 23-25 января, обе задачи `success`.
- Снимки SCD2 корректны: правила ДЗ 2.1, интервалы без пересечений, одна текущая версия, нет будущих событий.
- Исторический JOIN через BroadcastHashJoin, без дублей, совпадает с независимым расчётом и контрольными примерами.
- Метаполя заполнены и согласованы между задачами запуска; повторный расчёт не меняет бизнес-результат и другие даты.

## 8. Выгрузка результатов
Партиции обеих таблиц за три даты копируются из S3 в папку `results/` рядом с ноутбуком, для проверяющего.

In [ ]:
import os
sys.path.append("/opt/airflow/scripts")
from upload_data import s3_client   # клиент S3 из комплекта (те же ключи, что у Spark)

s3 = s3_client()
for table in ("user_scd2", "campaign_recipients"):
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket="hw23", Prefix=f"results/demo/{table}/"):
        for obj in page.get("Contents", []):
            local = os.path.join("results", obj["Key"].removeprefix("results/demo/"))
            os.makedirs(os.path.dirname(local), exist_ok=True)
            s3.download_file("hw23", obj["Key"], local)
            print(f"{obj['Size']:>8} B  {local}")

[2026-11-19T15:07:28.929+0000] {credentials.py:1147} INFO - Found credentials in environment variables.
       0 B  results/user_scd2/snapshot_date=2026-01-23/_SUCCESS
   36613 B  results/user_scd2/snapshot_date=2026-01-23/part-00000-220d6847-ac11-4a9b-bf29-618d6a8ec50e-c000.snappy.parquet
       0 B  results/user_scd2/snapshot_date=2026-01-24/_SUCCESS
   37643 B  results/user_scd2/snapshot_date=2026-01-24/part-00000-3e92ebd6-6894-4ccf-91eb-69fb3680b12c-c000.snappy.parquet
       0 B  results/user_scd2/snapshot_date=2026-01-25/_SUCCESS
